# GEDI Footprint Analysis Workflow (2 of 2 — Analysis)

Picks up where `gedi_footprint_1_extract.ipynb` left off. Instead of re-running the GEDI search/filter/clip steps, this notebook loads the **clipped points and footprints GeoJSON files** that notebook 1 saved.

Run the two cells below first (imports, then the file picker), then continue through the Monte Carlo simulation and CHM comparison sections in order.

In [ ]:
#------------ imports ----------------------------------------------------------
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog


---
## Load Saved GEDI Data

Select the **clipped points GeoJSON** file that notebook 1 saved (named like `<header>_gedi_shots_clipped.geojson`). A file-picker dialog will open. This notebook will then automatically look for the matching clipped footprints file (`<header>_gedi_footprints_clipped.geojson`) in the same folder — if it can't find it, you'll be asked to pick that one too.

In [ ]:
# ── Select and load the saved, clipped GEDI outputs from notebook 1 ────────────
def select_file(title, filetypes):
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    path = filedialog.askopenfilename(title=title, filetypes=filetypes)
    root.destroy()
    if not path:
        raise SystemExit(f'No file selected for: {title}')
    return path

print('Select the CLIPPED POINTS GeoJSON file saved by notebook 1')
print("(named like '<header>_gedi_shots_clipped.geojson')")
clipped_points_path = select_file(
    'Select clipped GEDI points GeoJSON',
    [('GeoJSON files', '*.geojson'), ('All files', '*.*')]
)

data_folder = os.path.dirname(clipped_points_path)
fname = os.path.basename(clipped_points_path)
file_header = fname.replace('_gedi_shots_clipped.geojson', '')

# Try to auto-locate the matching clipped footprints file in the same folder
guess_footprints_path = os.path.join(data_folder, f'{file_header}_gedi_footprints_clipped.geojson')

if os.path.exists(guess_footprints_path):
    clipped_footprints_path = guess_footprints_path
    print(f'\nAuto-found matching footprints file: {clipped_footprints_path}')
else:
    print('\nCould not auto-locate the matching footprints file — please select it.')
    clipped_footprints_path = select_file(
        'Select clipped GEDI footprints GeoJSON',
        [('GeoJSON files', '*.geojson'), ('All files', '*.*')]
    )

gdf_points_clipped = gpd.read_file(clipped_points_path)
gdf_footprints_clipped = gpd.read_file(clipped_footprints_path)

# Ensure shot_number stays as string (ArcGIS Pro compatibility, and needed
# for merges further down)
gdf_points_clipped['shot_number'] = gdf_points_clipped['shot_number'].astype(str)
gdf_footprints_clipped['shot_number'] = gdf_footprints_clipped['shot_number'].astype(str)

# Drop any leftover reading_order_id from a previous run so Step A below
# regenerates it cleanly
gdf_points_clipped = gdf_points_clipped.drop(columns='reading_order_id', errors='ignore')
gdf_footprints_clipped = gdf_footprints_clipped.drop(columns='reading_order_id', errors='ignore')

WGS84 = 'EPSG:4326'

print(f'\nLoaded:')
print(f'  data_folder = {data_folder}')
print(f'  file_header = {file_header}')
print(f'  Points     : {len(gdf_points_clipped):,} rows from {clipped_points_path}')
print(f'  Footprints : {len(gdf_footprints_clipped):,} rows from {clipped_footprints_path}')

## DONT RUN FOR NOW (Messing Around with Graph Ouputs for GEDI Data)

In [ ]:
# # ── RH Canopy Profile — One Chart Per Footprint ───────────────────────────────
# # Horizontal bar chart showing each rh metric as a canopy height for that footprint.
# # One figure saved per footprint. UAV CHM overlay to be added later.

# import matplotlib.pyplot as plt
# import math

# rh_cols        = ['rh25', 'rh50', 'rh75', 'rh95', 'rh98', 'rh99', 'rh100']
# rh_percentiles = [25,      50,     75,     95,     98,     99,     100]

# for plot_idx, (_, row) in enumerate(gdf_points_clipped.iterrows()):

#     fig, ax = plt.subplots(figsize=(6, 7))

#     rh_values = [row[col] for col in rh_cols]

#     # Height intervals between consecutive rh percentiles
#     heights   = [0] + rh_values
#     intervals = [heights[i+1] - heights[i] for i in range(len(rh_values))]
#     percentile_widths = [rh_percentiles[0]] + [
#         rh_percentiles[i] - rh_percentiles[i-1] for i in range(1, len(rh_percentiles))
#     ]

#     ax.barh(
#         y      = [heights[i] + intervals[i] / 2 for i in range(len(intervals))],
#         width  = percentile_widths,
#         height = intervals,
#         color='steelblue', edgecolor='white', linewidth=0.5, alpha=0.75,
#         label='GEDI RH profile'
#     )

#     # Label each rh metric
#     for rh_val, rh_pct in zip(rh_values, rh_percentiles):
#         ax.axhline(rh_val, color='black', linewidth=0.6, linestyle='--', alpha=0.4)
#         ax.text(max(percentile_widths) * 0.98, rh_val + 0.2,
#                 f'rh{rh_pct} = {rh_val:.1f} m',
#                 fontsize=7, ha='right', va='bottom')

#     ax.set_xlabel('Percentile interval width (%)', fontsize=9)
#     ax.set_ylabel('Height (m)', fontsize=9)
#     ax.set_ylim(0, max(rh_values) * 1.15)
#     ax.set_title(
#         f'Footprint {plot_idx + 1} — Canopy Height Profile\n'
#         f'Date: {row["date"]}  |  Beam: {row["beam"]}\n'
#         f'rh98: {row["rh98"]:.1f} m  |  Sensitivity: {row["sensitivity"]:.2f}',
#         fontsize=9
#     )
#     ax.legend(fontsize=8)
#     ax.grid(axis='x', linestyle='--', alpha=0.3)
#     ax.tick_params(labelsize=8)

#     plt.tight_layout()

#     #optional for saving the graphs created
#     #save_path = os.path.join(data_folder, f'{file_header}_footprint{plot_idx + 1}_rh_profile.png')
#     #plt.savefig(save_path, dpi=150, bbox_inches='tight')
#     #print(f"Saved: {save_path}")

#     plt.show()

#     # ── Raw GEDI Waveform Plot — One Chart Per Footprint ─────────────────────────
# # Plots the raw received waveform (rxwaveform) for each clipped footprint.
# # The waveform shows laser return intensity vs sample index (proxy for height) —
# # peaks indicate surfaces where the laser reflected (ground, understory, canopy).
# # This is the black line from the reference figure.

# for plot_idx, (_, row) in enumerate(gdf_points_clipped.iterrows()):

#     sn = row['shot_number']

#     if sn not in waveform_store:
#         print(f"Footprint {plot_idx + 1}: no waveform found for shot {sn}")
#         continue

#     waveform = waveform_store[sn]

#     # Sample index runs top-down (first sample = top of atmosphere,
#     # last sample = ground), so we flip it so ground is at the bottom
#     samples = np.arange(len(waveform))
#     samples_flipped = samples[::-1]

#     fig, ax = plt.subplots(figsize=(5, 7))

#     ax.plot(
#         waveform, samples_flipped,
#         color='black', linewidth=1.2, label='GEDI waveform'
#     )

#     # Shade the area under the waveform like the reference figure
#     ax.fill_betweenx(
#         samples_flipped, 0, waveform,
#         color='steelblue', alpha=0.2
#     )

#     ax.set_xlabel('Return intensity (counts)', fontsize=9)
#     ax.set_ylabel('Sample index (↑ = higher in canopy)', fontsize=9)
#     ax.set_title(
#         f'Footprint {plot_idx + 1} — Raw GEDI Waveform\n'
#         f'Date: {row["date"]}  |  Beam: {row["beam"]}\n'
#         f'rh98: {row["rh98"]:.1f} m  |  Sensitivity: {row["sensitivity"]:.2f}',
#         fontsize=9
#     )
#     ax.legend(fontsize=8)
#     ax.grid(axis='x', linestyle='--', alpha=0.3)
#     ax.tick_params(labelsize=8)

#     plt.tight_layout()
#     plt.show()

## Create Labels Easily Readable for Labeling Footprints

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP A — Assign reading-order numbers (top-to-bottom, left-to-right)
# ════════════════════════════════════════════════════════════════════════════
# Groups points into "rows" based on latitude proximity, then numbers each
# row left-to-right (low lon → high lon), rows ordered top-to-bottom (high lat → low lat)

ROW_TOLERANCE_M = 15.0   # points within this many metres of latitude count as the same "row"
                          # tune this based on your point spacing — too small splits a row
                          # into multiple rows, too large merges separate rows together

row_tol_deg = ROW_TOLERANCE_M / 111320.0   # ~111,320 m per degree latitude

# Ensure shot_number is a consistent string type before any merging
gdf_points_clipped['shot_number'] = gdf_points_clipped['shot_number'].astype(str)

gdf_sorted = gdf_points_clipped.sort_values('lat', ascending=False).reset_index(drop=True)

row_ids = []
current_row = 0
row_ref_lat = gdf_sorted.loc[0, 'lat']

for lat_val in gdf_sorted['lat']:
    if row_ref_lat - lat_val > row_tol_deg:
        current_row += 1
        row_ref_lat = lat_val
    row_ids.append(current_row)

gdf_sorted['row_id'] = row_ids
gdf_sorted = gdf_sorted.sort_values(['row_id', 'lon'], ascending=[True, True]).reset_index(drop=True)
gdf_sorted['reading_order_id'] = gdf_sorted.index + 1   # start at 1, not 0

gdf_points_clipped = gdf_points_clipped.merge(
    gdf_sorted[['shot_number', 'reading_order_id']],
    on='shot_number',
    how='left'
)

n_missing_a = gdf_points_clipped['reading_order_id'].isna().sum()
if n_missing_a > 0:
    print(f"WARNING: {n_missing_a} points in gdf_points_clipped have no reading_order_id — check shot_number dtypes")

print(gdf_points_clipped[['shot_number', 'lat', 'lon', 'reading_order_id']]
      .sort_values('reading_order_id').to_string(index=False))
# ════════════════════════════════════════════════════════════════════════════
# STEP A.1 — Propagate reading_order_id onto footprints, re-save GeoJSON files
# ════════════════════════════════════════════════════════════════════════════
# gdf_footprints_clipped doesn't have reading_order_id yet — it was only
# assigned to gdf_points_clipped in Step A. Merge it across using shot_number,
# then overwrite both clipped GeoJSON files so the attribute table includes
# the new column when opened in QGIS/ArcGIS.

gdf_footprints_clipped['shot_number'] = gdf_footprints_clipped['shot_number'].astype(str)

if 'reading_order_id' in gdf_footprints_clipped.columns:
    gdf_footprints_clipped = gdf_footprints_clipped.drop(columns='reading_order_id')

reading_id_lookup = gdf_points_clipped[['shot_number', 'reading_order_id']].drop_duplicates()

gdf_footprints_clipped = gdf_footprints_clipped.merge(
    reading_id_lookup, on='shot_number', how='left'
)

n_missing_fp = gdf_footprints_clipped['reading_order_id'].isna().sum()
if n_missing_fp > 0:
    print(f"WARNING: {n_missing_fp} footprints did not get a reading_order_id")
else:
    print("reading_order_id successfully added to gdf_footprints_clipped")

# Re-save both clipped GeoJSON files, now including reading_order_id
gdf_footprints_clipped.to_file(clipped_footprints_path, driver='GeoJSON')
gdf_points_clipped.to_file(clipped_points_path, driver='GeoJSON')

print(f"Footprints GeoJSON updated: {clipped_footprints_path}")
print(f"Points GeoJSON updated:     {clipped_points_path}")

## Step 10: Monte Carlo Simulation of New Footprint Center Coordinate Locations
==========================================================================================

Based on:
  - 'Simulation-Based Correction of Geolocation Errors in GEDI Footprint Positions
     Using Monte Carlo Approach' (Wang et. al 2025)
  - 'The impact of geolocation uncertainty on GEDI tropical forest canopy height
     estimation and change monitoring' (Roy et al. 2021)

Method:
  Each GEDI footprint location is shifted with randomly generated position errors
  modelled using the GEDI geolocation uncertainty (Dubayah et al., 2020a):

      x*_i = x + s_i * cos(theta_i)
      y*_i = y + s_i * sin(theta_i)

  where:
    (x*_i, y*_i) = shifted GEDI footprint center coordinate
    (x, y)       = GEDI product reported footprint center coordinate
    s_i          ~ N(mu=0 m, sigma=10 m)  [geolocation uncertainty]
    theta_i      ~ Uniform(0, 2*pi)       [random direction]
    n            = 300 simulations per footprint

Includes Printout of Map of GEDI Footprints and Simulated Centerpoints and location Distribution of Simulated Center Coordinates

In [ ]:
# ── Simulation parameters ────────────────────────────────────────────────────
N_SIMULATIONS = 300       # number of random shifted positions generated per footprint
SIGMA_M       = 10.0      # GEDI geolocation uncertainty: 1 standard deviation = 10 m
                          # (Dubayah et al. 2020a) — most shots land within 10 m of true position
MU_M          = 0.0       # zero-mean error: no systematic bias assumed, errors are random
SEED          = None        # fixing the seed makes results reproducible run-to-run
                          # remove or change this for true randomness in production
rng           = np.random.default_rng(SEED)

# ── Coordinate reference systems ──────────────────────────────────────────────
# GEDI reports positions in WGS84 (degrees), but we need to apply metre-scale
# offsets. We temporarily project to UTM (metres) to do the geometry, then
# project back to WGS84 for the output.
WGS84      = 'EPSG:4326'
METRIC_CRS = 'EPSG:32610'    # UTM Zone 10N — covers central California
                              # change this if your study area is in a different UTM zone

to_metric = Transformer.from_crs(WGS84, METRIC_CRS, always_xy=True)
to_wgs84  = Transformer.from_crs(METRIC_CRS, WGS84,  always_xy=True)

print(f"Input footprints : {len(gdf_points_clipped):,}")
print(f"Monte Carlo n    : {N_SIMULATIONS}")
print(f"Geolocation σ    : {SIGMA_M} m  (μ = {MU_M} m)")

# ── Core Monte Carlo loop ─────────────────────────────────────────────────────
# For each real GEDI footprint, we simulate N_SIMULATIONS possible "true"
# positions by applying random position errors. This models the uncertainty
# in where the laser actually hit the ground vs where GEDI says it did.
records = []

for fp_idx, row in gdf_points_clipped.iterrows():

    footprint_id = row['shot_number']


    # Step 1: get the reported WGS84 position and convert to metres (UTM)
    lon_orig = row.geometry.x
    lat_orig = row.geometry.y
    x_m, y_m = to_metric.transform(lon_orig, lat_orig)

    # Step 2: sample random radial displacements
    # s_i ~ N(0, 10m) — how far each simulated point is shifted from the original
    # This follows a normal distribution: most shifts are small, few are large
    s_i = rng.normal(loc=MU_M, scale=SIGMA_M, size=N_SIMULATIONS)

    # Step 3: sample random azimuth directions
    # theta_i ~ Uniform(0, 2π) — the direction of each shift is completely random
    # This means errors are equally likely in any compass direction
    theta_i = rng.uniform(low=0.0, high=2 * np.pi, size=N_SIMULATIONS)

    # Step 4: apply the offsets in metric space
    # x*_i = x + s_i * cos(theta_i)   (east-west shift)
    # y*_i = y + s_i * sin(theta_i)   (north-south shift)
    x_star = x_m + s_i * np.cos(theta_i)
    y_star = y_m + s_i * np.sin(theta_i)

    # Step 5: project the shifted positions back to WGS84 (degrees)
    lon_star, lat_star = to_wgs84.transform(x_star, y_star)

    # Step 6: store each simulation as a row in the results list
    for sim_idx in range(N_SIMULATIONS):
        rec = {
            'footprint_id'   : footprint_id,              # which original footprint this belongs to
            'simulation_id'  : sim_idx + 1,           # which simulation iteration (1–300)
            'lon_original'   : lon_orig,              # original reported GEDI position
            'lat_original'   : lat_orig,
            's_i_m'          : s_i[sim_idx],          # radial displacement magnitude in metres
            'theta_i_deg'    : np.degrees(theta_i[sim_idx]),  # shift direction in degrees
            'dx_m'           : s_i[sim_idx] * np.cos(theta_i[sim_idx]),  # east-west component
            'dy_m'           : s_i[sim_idx] * np.sin(theta_i[sim_idx]),  # north-south component
            'lon_shifted'    : lon_star[sim_idx],     # shifted position (the simulated true location)
            'lat_shifted'    : lat_star[sim_idx],
            'geometry'       : Point(lon_star[sim_idx], lat_star[sim_idx]),
        }
        records.append(rec)

# ── Assemble into a single GeoD98
# 
# ataFrame ──────────────────────────────────────
# Result: one row per simulation per footprint
# e.g. 6 footprints × 300 simulations = 1,800 rows
gdf_mc = gpd.GeoDataFrame(records, geometry='geometry', crs=WGS84)

print(f"\nMonte Carlo output shape : {gdf_mc.shape}")
print(f"  ({len(gdf_points_clipped)} footprints × {N_SIMULATIONS} simulations = "
      f"{len(gdf_points_clipped) * N_SIMULATIONS:,} rows)")

# ── Per-footprint summary statistics ─────────────────────────────────────────
# Collapse the 300 simulations per footprint into summary stats:
#   mean shifted position → best estimate of the "true" corrected location
#   std of shifted positions → how uncertain the corrected position is
#   mean radial displacement → average error magnitude across all simulations
summary = (
    gdf_mc
    .groupby('footprint_id')
    .agg(
        lon_original       = ('lon_original',  'first'),
        lat_original       = ('lat_original',  'first'),
        lon_shifted_mean   = ('lon_shifted',   'mean'),   # ensemble mean corrected longitude
        lat_shifted_mean   = ('lat_shifted',   'mean'),   # ensemble mean corrected latitude
        lon_shifted_std    = ('lon_shifted',   'std'),    # spread in longitude across simulations
        lat_shifted_std    = ('lat_shifted',   'std'),    # spread in latitude across simulations
        mean_radial_disp_m = ('s_i_m',         lambda x: np.abs(x).mean()),  # avg displacement
        n_simulations      = ('simulation_id', 'count'),
    )
    .reset_index()
)

# Attach the ensemble-mean corrected position as the geometry
summary_geom = [
    Point(row.lon_shifted_mean, row.lat_shifted_mean)
    for _, row in summary.iterrows()
]
gdf_summary = gpd.GeoDataFrame(summary, geometry=summary_geom, crs=WGS84)

print("\nPer-footprint summary (first 5 rows):")
display(gdf_summary.head())

# STEP E — Append reading_order_id to gdf_mc and gdf_summary
# ════════════════════════════════════════════════════════════════════════════
if 'reading_order_id' not in gdf_points_clipped.columns:
    raise RuntimeError(
        "reading_order_id missing from gdf_points_clipped — Step A must run "
        "successfully before this step. Re-run from the top of the notebook."
    )

reading_order_lookup = gdf_points_clipped[['shot_number', 'reading_order_id']].drop_duplicates()
reading_order_lookup['shot_number'] = reading_order_lookup['shot_number'].astype(str)

gdf_mc['footprint_id'] = gdf_mc['footprint_id'].astype(str)
gdf_summary['footprint_id'] = gdf_summary['footprint_id'].astype(str)

gdf_mc = gdf_mc.merge(
    reading_order_lookup, left_on='footprint_id', right_on='shot_number', how='left'
).drop(columns='shot_number')

gdf_summary = gdf_summary.merge(
    reading_order_lookup, left_on='footprint_id', right_on='shot_number', how='left'
).drop(columns='shot_number')

n_missing_e = gdf_mc['reading_order_id'].isna().sum()
if n_missing_e > 0:
    print(f"WARNING: {n_missing_e} rows in gdf_mc have no matching reading_order_id")
else:
    print("reading_order_id successfully appended to gdf_mc and gdf_summary")


# ════════════════════════════════════════════════════════════════════════════
# STEP F — Export results to Excel for inspection/editing
# ════════════════════════════════════════════════════════════════════════════
excel_path = os.path.join(data_folder, f'{file_header}_monte_carlo_results.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    gdf_mc.drop(columns='geometry').to_excel(writer, sheet_name='all_simulations', index=False)
    gdf_summary.drop(columns='geometry').to_excel(writer, sheet_name='per_footprint_summary', index=False)

print(f"Excel file saved to: {excel_path}")


# ── Diagnostic plots ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: shows the cloud of 300 shifted positions around each original footprint
# Each cluster of blue dots represents the uncertainty envelope for one GEDI shot
ax = axes[0]
ax.scatter(
    gdf_mc['lon_shifted'], gdf_mc['lat_shifted'],
    s=1, alpha=0.15, color='steelblue', label='Shifted positions'
)
ax.scatter(
    gdf_points_clipped.geometry.x, gdf_points_clipped.geometry.y,
    s=40, color='red', zorder=5, label='Original reported positions'
)
# Label each original point with its reading-order number
for _, row in gdf_points_clipped.iterrows():
    ax.annotate(
        str(int(row['reading_order_id'])),
        xy=(row.geometry.x, row.geometry.y),
        xytext=(6, 6),
        textcoords='offset points',
        fontsize=11,
        fontweight='bold',
        color='black',
        zorder=6,
    )
    
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_title(f'Monte Carlo Shifted Footprint Positions\n'
             f'(n={N_SIMULATIONS} per footprint, σ={SIGMA_M} m)')
ax.legend(markerscale=3, fontsize=8)

# Right panel: histogram of how large the random shifts were
# Should look like a half-normal distribution centred near 0
# The red line marks the 10 m 1-sigma threshold
ax2 = axes[1]
ax2.hist(np.abs(gdf_mc['s_i_m']), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(SIGMA_M, color='red', linestyle='--', linewidth=1.5, label=f'σ = {SIGMA_M} m')
ax2.set_xlabel('|Radial displacement| (m)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Simulated Radial Displacements')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(data_folder, f'{file_header}_monte_carlo.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

print("\nOutputs ready:")
print("  gdf_mc      – full Monte Carlo simulation GeoDataFrame (all footprints × all iterations)")
print("  gdf_summary – per-footprint ensemble summary with mean-corrected positions")

## Plot shifted footprint circles for each original footprint 
Replicates methods of "The impact of geolocation uncertainty on GEDI tropical forest canopy height estimation and change monitoring" (Roy et al. 2021)  
Prints out simulated buffered footprints in reference to the original footprint location  
Seeds at 42, can set to true random by setting Seed = None  
 - red circle = original reported GEDI 25m footprint
 - black circles = 300 simulated shifted 25m footprints

In [ ]:
# ── Plot shifted footprint circles for each original footprint ────────────────
# Replicates the style of panel (b) in the paper:
#   - red circle = original reported GEDI 25m footprint
#   - black circles = 300 simulated shifted 25m footprints

from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import math

n_footprints = len(gdf_points_clipped)
n_cols = 3
n_rows = math.ceil(n_footprints / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

FOOTPRINT_RADIUS_M = 12.5   # 25m diameter footprint → 12.5m radius


for plot_idx, (fp_idx, fp_sims) in enumerate(gdf_mc.groupby('footprint_id')):

    ax = axes[plot_idx]

    # Get original position in metres (UTM)
    lon_orig = fp_sims['lon_original'].iloc[0]
    lat_orig = fp_sims['lat_original'].iloc[0]
    x_orig, y_orig = to_metric.transform(lon_orig, lat_orig)

    # Draw each of the 300 shifted 25m circles in black
    for _, sim_row in fp_sims.iterrows():
        x_shift, y_shift = to_metric.transform(sim_row['lon_shifted'], sim_row['lat_shifted'])
        circle = plt.Circle(
            (x_shift, y_shift), FOOTPRINT_RADIUS_M,
            color='black', fill=False, linewidth=0.3, alpha=0.4
        )
        ax.add_patch(circle)

    # Draw the original reported footprint in red on top
    original_circle = plt.Circle(
        (x_orig, y_orig), FOOTPRINT_RADIUS_M,
        color='red', fill=False, linewidth=1.5
    )
    ax.add_patch(original_circle)

    # Set axis limits with padding around the footprint cloud
    pad = FOOTPRINT_RADIUS_M + SIGMA_M * 3   # show out to ~3σ
    ax.set_xlim(x_orig - pad, x_orig + pad)
    ax.set_ylim(y_orig - pad, y_orig + pad)
    ax.set_aspect('equal')

    ax.set_xlabel('UTM Easting (m)', fontsize=8)
    ax.set_ylabel('UTM Northing (m)', fontsize=8)
    ax.set_title(f'Footprint {plot_idx + 1}  (shot {fp_idx})', fontsize=9)
    ax.tick_params(labelsize=7)

plt.suptitle(
    f'Monte Carlo Simulated Footprint Positions\n'
    f'(n={N_SIMULATIONS} per footprint, σ={SIGMA_M} m, red = original reported position)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(
    os.path.join(data_folder, f'{file_header}_monte_carlo_circles.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print("Figure saved.")

## Generate P95 from the UAV CHM  
Reads the GeoTIFF file from the previously run R code that generates the CHM  
* For each of the 6 footprints, for each of the 300 iterrations, it takes the value below which 95% of the CHM heights in the buffer fall  
* This is saved to gdf_mc data frame as rh95_uav_chm  
* This information is then saved to an excel sheet titled: {file_header}_monte_carlo_results.xlsx  

    Sheet 1: all_simulations ----------> contains rh95_uav_chm for each footprint  
    Sheet 2: per_footprint_summary -> contains a summary table with variation of 300 simulated P95 defined by the IQR (75th - 25th percentiles)  
    which provides insights into the heterogeneity of forests at the footprint locations
    (The IQR accounts for 50% of the variation in the forest canopy height due to GEDI geolocation uncertainty. High IQR values, greater than or comparable to the relative height derived from the ALS data at the GEDI reported footprint location are shown to occur where the footprint covered or was adjacent to spatially heterogeneous canopies, including canopies with small forest stands, holes in the vegetation canopy, and forest edges.)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP H — Extract RH95 from UAV CHM at each simulated footprint center
# ════════════════════════════════════════════════════════════════════════════
# For each simulated coordinate (one of the 300 Monte Carlo positions per
# footprint), extract a 12.5 m radius circular buffer (matching GEDI's 25 m
# footprint diameter) from the UAV-derived Canopy Height Model (CHM) GeoTIFF,
# and compute the 95th percentile of CHM pixel values within that buffer as
# the UAV-equivalent RH95 metric for that simulated position.
#
# Methodology reference:
#   Roy et al. (2021), "The impact of geolocation uncertainty on GEDI
#   tropical forest canopy height estimation and change monitoring",
#   extracts relative-height percentiles (h95, h85, h75) from reference canopy data
#   at each simulated position to quantify geolocation-driven RH variability.

import rasterio
from rasterio.windows import from_bounds
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
import numpy as np

# ── Prompt user for the CHM GeoTIFF path ──────────────────────────────────────
chm_path = input("Enter the full path to the UAV CHM GeoTIFF file: ").strip()

if not os.path.exists(chm_path):
    raise FileNotFoundError(f"CHM file not found at: {chm_path}")

BUFFER_RADIUS_M = 12.5   # 25 m GEDI footprint diameter → 12.5 m radius buffer

print(f"\nUsing CHM file: {chm_path}")
print(f"Buffer radius  : {BUFFER_RADIUS_M} m")

with rasterio.open(chm_path) as chm_src:
    chm_crs = chm_src.crs
    chm_nodata = chm_src.nodata

    print(f"CHM CRS        : {chm_crs}")
    print(f"CHM nodata     : {chm_nodata}")
    print(f"CHM resolution : {chm_src.res}")

    # Reproject simulated points from WGS84 into the CHM's native CRS so the
    # 12.5 m buffer radius is applied correctly in real metres, not degrees
    to_chm_crs = Transformer.from_crs(WGS84, chm_crs, always_xy=True)

    rh95_uav = np.full(len(gdf_mc), np.nan)

    n_total = len(gdf_mc)
    n_outside = 0
    n_nodata = 0

    # main loop for getting raster info within each simulated footprint buffer
    for i, (lon, lat) in enumerate(zip(gdf_mc['lon_shifted'], gdf_mc['lat_shifted'])):

        x_chm, y_chm = to_chm_crs.transform(lon, lat) # converts each points coordinates into the CHM's CRS 
        buffer_geom = Point(x_chm, y_chm).buffer(BUFFER_RADIUS_M) # draws a 12.5m buffer around each point

        try:
            out_image, out_transform = rio_mask( # clips the raster down to just the pixels within the circle (12.5m radius)
                chm_src, [mapping(buffer_geom)], crop=True, filled=True, nodata=chm_nodata
            )
        except ValueError:
            # Buffer falls entirely outside the raster extent
            n_outside += 1
            continue

        # filtering and computing the RH95 percentile
        chm_values = out_image[0]

        if chm_nodata is not None:
            valid_values = chm_values[chm_values != chm_nodata]
        else:
            valid_values = chm_values.flatten()

        # Also drop NaNs in case the raster uses NaN as nodata
        valid_values = valid_values[~np.isnan(valid_values)]

        if valid_values.size == 0:
            n_nodata += 1
            continue

        rh95_uav[i] = np.percentile(valid_values, 95) #actual RH95 calculation (value below which 95% of the CHm heights in the buffer fall)

        if (i + 1) % 500 == 0:
            print(f"  Processed {i + 1:,} / {n_total:,} simulated points...")

gdf_mc['rh95_uav_chm'] = rh95_uav

print(f"\nRH95 extraction complete.")
print(f"  Total simulated points        : {n_total:,}")
print(f"  Points outside CHM extent     : {n_outside:,}")
print(f"  Points with no valid CHM data : {n_nodata:,}")
print(f"  Points successfully extracted : {n_total - n_outside - n_nodata:,}")


# ── Roll the per-simulation RH95 values up into the per-footprint summary ────
rh95_summary = (
    gdf_mc
    .groupby('footprint_id')['rh95_uav_chm']
    .agg(
        rh95_uav_mean='mean',   # ensemble mean UAV RH95 across all simulations
        rh95_uav_std='std',     # spread in UAV RH95 due to geolocation uncertainty
        rh95_uav_p25=lambda x: np.nanpercentile(x, 25),  # matches paper's IQR approach
        rh95_uav_p75=lambda x: np.nanpercentile(x, 75),
    )
)
rh95_summary['rh95_uav_iqr'] = rh95_summary['rh95_uav_p75'] - rh95_summary['rh95_uav_p25']
rh95_summary = rh95_summary.reset_index()

gdf_summary = gdf_summary.merge(rh95_summary, on='footprint_id', how='left')

print("\nPer-footprint RH95 summary (first 5 rows):")
display(gdf_summary[['footprint_id', 'reading_order_id', 'rh95_uav_mean',
                      'rh95_uav_std', 'rh95_uav_iqr']].head())


# ════════════════════════════════════════════════════════════════════════════
# STEP I — Re-save updated results to the Excel file and GeoJSON
# ════════════════════════════════════════════════════════════════════════════
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    gdf_mc.drop(columns='geometry').to_excel(writer, sheet_name='all_simulations', index=False)
    gdf_summary.drop(columns='geometry').to_excel(writer, sheet_name='per_footprint_summary', index=False)

print(f"\nExcel file updated with RH95 data: {excel_path}")

## Comparing Simulated P95 Footprints to OG GEDI RH95 (RH95 - P95i, where i = 300)
* Finds which simulated UAV footprint data has the lowest error compared to the GEDI footprint data
* Saves this RH95 - P95 into {file_name}_monte_carlo_results.xlsx sheet in rh95_error and rh95_abs_error columns of all_simulations sheet  
* Does some updates to per_footprint_summary and adds best_matches sheet as well (this may need to be checked in the future)

Then: Next cell block spits out visual representation of original footprint location and "best match" location

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP J — Compare UAV-derived RH95 to original GEDI RH95, find best match
# ════════════════════════════════════════════════════════════════════════════
# For each simulated position, compute the error between the UAV CHM-derived
# RH95 (Step H) and the GEDI-reported RH95 at that footprint. Then, for each
# footprint, identify which of the 300 simulated positions produced the
# closest match — this is effectively asking "where would the laser have
# needed to actually land for the UAV ground truth to agree with what GEDI
# reported?"

# ── Step 1: attach the original GEDI RH95 value to every simulation row ──────
gedi_rh95_lookup = gdf_points_clipped[['shot_number', 'rh95']].drop_duplicates()
gedi_rh95_lookup = gedi_rh95_lookup.rename(columns={'rh95': 'rh95_gedi'})
gedi_rh95_lookup['shot_number'] = gedi_rh95_lookup['shot_number'].astype(str)

gdf_mc = gdf_mc.merge(
    gedi_rh95_lookup, left_on='footprint_id', right_on='shot_number', how='left'
).drop(columns='shot_number')

# ── Step 2: compute error for every simulation ────────────────────────────────
gdf_mc['rh95_error'] = gdf_mc['rh95_uav_chm'] - gdf_mc['rh95_gedi']        # signed error
gdf_mc['rh95_abs_error'] = gdf_mc['rh95_error'].abs()                     # absolute error

# ── Step 3: overall error statistics across ALL simulations ──────────────────
valid_errors = gdf_mc['rh95_error'].dropna()

print("═" * 60)
print("OVERALL RH95 ERROR STATISTICS (UAV CHM vs GEDI-reported)")
print("═" * 60)
print(f"  n (valid comparisons) : {len(valid_errors):,}")
print(f"  Mean error (bias)     : {valid_errors.mean():.3f} m")
print(f"  Std of error          : {valid_errors.std():.3f} m")
print(f"  MAE                   : {valid_errors.abs().mean():.3f} m")
print(f"  RMSE                  : {np.sqrt((valid_errors**2).mean()):.3f} m")
print(f"  Min error              : {valid_errors.min():.3f} m")
print(f"  Max error              : {valid_errors.max():.3f} m")

# ── Step 4: per-footprint best match (lowest absolute error) ─────────────────
best_match_idx = gdf_mc.groupby('footprint_id')['rh95_abs_error'].idxmin()
best_matches = gdf_mc.loc[best_match_idx].copy()
best_matches = best_matches.sort_values('reading_order_id')

print("\n" + "═" * 60)
print("BEST-MATCHING SIMULATED POSITION PER FOOTPRINT")
print("═" * 60)
display(best_matches[[
    'reading_order_id', 'footprint_id', 'simulation_id',
    'lon_original', 'lat_original', 'lon_shifted', 'lat_shifted',
    'rh95_gedi', 'rh95_uav_chm', 'rh95_error', 'rh95_abs_error'
]])

# Identify the single overall best match across all footprints
overall_best = best_matches.loc[best_matches['rh95_abs_error'].idxmin()]
print(f"\nBest overall match: footprint reading_order_id "
      f"{int(overall_best['reading_order_id'])} "
      f"(shot {overall_best['footprint_id']}), "
      f"simulation #{int(overall_best['simulation_id'])}, "
      f"abs error = {overall_best['rh95_abs_error']:.3f} m")

# ── Step 5: append best-match info back onto gdf_summary ─────────────────────
gdf_summary = gdf_summary.merge(
    best_matches[['footprint_id', 'lon_shifted', 'lat_shifted',
                  'rh95_gedi', 'rh95_uav_chm', 'rh95_error', 'rh95_abs_error']]
    .rename(columns={
        'lon_shifted': 'best_match_lon',
        'lat_shifted': 'best_match_lat',
        'rh95_error': 'best_match_error',
        'rh95_abs_error': 'best_match_abs_error',
    }),
    on='footprint_id', how='left'
)

# ── Step 6: re-save Excel with the new comparison data ────────────────────────
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    gdf_mc.drop(columns='geometry').to_excel(writer, sheet_name='all_simulations', index=False)
    gdf_summary.drop(columns='geometry').to_excel(writer, sheet_name='per_footprint_summary', index=False)
    best_matches.drop(columns='geometry').to_excel(writer, sheet_name='best_matches', index=False)

print(f"\nExcel file updated with RH95 comparison data: {excel_path}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP K — Grid plot: original vs. best-matching buffer, per footprint
# ════════════════════════════════════════════════════════════════════════════
from rasterio.windows import from_bounds as window_from_bounds
from matplotlib.patches import Circle

n_footprints = len(best_matches)
n_cols = min(3, n_footprints)
n_rows = int(np.ceil(n_footprints / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5.5 * n_rows))
axes = np.atleast_1d(axes).flatten()

with rasterio.open(chm_path) as chm_src:
    chm_crs = chm_src.crs
    to_chm_crs = Transformer.from_crs(WGS84, chm_crs, always_xy=True)

    for ax_idx, (_, fp) in enumerate(best_matches.iterrows()):
        ax = axes[ax_idx]

        # Convert both the original and best-match points into CHM CRS
        x_orig, y_orig = to_chm_crs.transform(fp['lon_original'], fp['lat_original'])
        x_best, y_best = to_chm_crs.transform(fp['lon_shifted'], fp['lat_shifted'])

        # Build a display window that comfortably contains both buffers
        pad = BUFFER_RADIUS_M * 2.5
        min_x = min(x_orig, x_best) - pad
        max_x = max(x_orig, x_best) + pad
        min_y = min(y_orig, y_best) - pad
        max_y = max(y_orig, y_best) + pad

        window = window_from_bounds(min_x, min_y, max_x, max_y, transform=chm_src.transform)
        chm_crop = chm_src.read(1, window=window)
        crop_transform = chm_src.window_transform(window)

        # Display extent for imshow, matching the cropped raster's bounds
        extent = (
            crop_transform.c,
            crop_transform.c + chm_crop.shape[1] * crop_transform.a,
            crop_transform.f + chm_crop.shape[0] * crop_transform.e,
            crop_transform.f,
        )

        im = ax.imshow(chm_crop, extent=extent, cmap='YlGn', origin='upper')

        # Original footprint buffer (red)
        ax.add_patch(Circle((x_orig, y_orig), BUFFER_RADIUS_M,
                             fill=False, edgecolor='red', linewidth=2, label='Original'))
        ax.plot(x_orig, y_orig, 'r+', markersize=10, markeredgewidth=2)

        # Best-matching simulated buffer (blue)
        ax.add_patch(Circle((x_best, y_best), BUFFER_RADIUS_M,
                             fill=False, edgecolor='blue', linewidth=2, label='Best match'))
        ax.plot(x_best, y_best, 'b+', markersize=10, markeredgewidth=2)

        ax.set_title(
            f"#{int(fp['reading_order_id'])}  |  "
            f"GEDI RH95={fp['rh95_gedi']:.2f}m  UAV RH95={fp['rh95_uav_chm']:.2f}m\n"
            f"error={fp['rh95_error']:+.2f}m",
            fontsize=10
        )
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
        ax.legend(fontsize=7, loc='upper right')
        ax.set_aspect('equal')

# Hide any unused subplot axes if grid is not fully filled
for ax_idx in range(n_footprints, len(axes)):
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(data_folder, f'{file_header}_best_match_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Best-match grid figure saved.")

## Code for Gridded Center Coordinate Shifts
following protocol from Mountains of Error Paper


In [ ]:
# for the original footprint locations, get P95, P98, P99, and P100 from the UAV CHM and compare it to the OG GEDI footprint RH metrics via RH95-P95, RH95-P98, RH95-P99, and RH95-P100. This will be done by creating a new column in the gdf_summary dataframe for each of these comparisons.
# from Mountains of error: For each percentile pair, we summarized the distributions of GEDI RH and UAV LiDAR percentiles with overlapping histograms and compared their medians to assess systematic offsets. At the footprint level, we calculated bias (mean error), median error, mean absolute error (MAE), root mean square error (RMSE), and Pearson’s correlation coefficient, and fit simple linear regression models with UAV LiDAR percentiles as the predictor. These diagnostics were used to identify the percentile pairing that best matched the UAV LiDAR reference. On the basis of this comparison, RH95 versus UAV P95 showed the smallest systematic offset and was therefore selected as the primary metric for subsequent footprint-scale analyses. All subsequent error analyses used canopy height error defined as  ε=HRH95 HUAV,P95,  computed for each footprint. We summarized the distribution of ε with histograms and descriptive statistics (bias, median, RMSE, and selected quantiles), and quantified the proportions of footprints falling within fixed error thresholds as well as the fractions showing strong under- and overestimation. To evaluate the influence of topography on GEDI performance, mean slope was calculated using the DTM and then averaging the slope values within the footprint. We then related canopy height error (ε) to slope by grouping footprints into slope classes and summarizing the distribution of ε within each class. Furthermore, to disentangle covarying terrain and vegetation controls on canopy height error, we fitted (i) a robust multiple regression and (ii) a penalized generalized additive model (GAM) with predictors including mean elevation, mean slope, within-footprint relief, UAV P95 canopy height, and a canopycover proxy (fraction of CHM ≥ 2 m) for each footprint.

# ════════════════════════════════════════════════════════════════════════════
# STEP H2 — Compare GEDI RH95/RH98/RH99/RH100 to UAV CHM percentiles
#           at the ORIGINAL (unshifted) footprint coordinates
# ════════════════════════════════════════════════════════════════════════════
# For each GEDI footprint's original center coordinate, extract a 12.5 m
# radius circular buffer from the UAV CHM and compute the 95th, 98th, 99th,
# and 100th percentiles of CHM height within that buffer. These are compared
# directly against the corresponding GEDI RH95/RH98/RH99/RH100 values.

import rasterio
from rasterio.mask import mask as rio_mask
from shapely.geometry import Point, mapping
import numpy as np
import pandas as pd

# ── Prompt user for the CHM GeoTIFF path (reuse if already defined) ─────────
if 'chm_path' not in dir():
    chm_path = input("Enter the full path to the UAV CHM GeoTIFF file: ").strip()

if not os.path.exists(chm_path):
    raise FileNotFoundError(f"CHM file not found at: {chm_path}")

BUFFER_RADIUS_M = 12.5   # 25 m GEDI footprint diameter → 12.5 m radius buffer
PERCENTILES = [95, 98, 99, 100]  # RH95, RH98, RH99, RH100

print(f"\nUsing CHM file: {chm_path}")
print(f"Buffer radius  : {BUFFER_RADIUS_M} m")
print(f"Percentiles    : {PERCENTILES}")

# ── Assumes gdf_footprints holds ONE row per footprint with the ORIGINAL
#    (unshifted) lon/lat, plus the GEDI rh95/rh98/rh99/rh100 columns.
#    Adjust column names below if yours differ. ─────────────────────────────
LON_COL = 'lon'
LAT_COL = 'lat'
GEDI_COLS = {95: 'rh95', 98: 'rh98', 99: 'rh99', 100: 'rh100'}

with rasterio.open(chm_path) as chm_src:
    chm_crs = chm_src.crs
    chm_nodata = chm_src.nodata

    print(f"CHM CRS        : {chm_crs}")
    print(f"CHM nodata     : {chm_nodata}")
    print(f"CHM resolution : {chm_src.res}")

    to_chm_crs = Transformer.from_crs(WGS84, chm_crs, always_xy=True)

    n_total = len(gdf_footprints_clipped)
    uav_pct = {p: np.full(n_total, np.nan) for p in PERCENTILES}

    n_outside = 0
    n_nodata = 0

    for i, (lon, lat) in enumerate(zip(gdf_footprints_clipped[LON_COL], gdf_footprints_clipped[LAT_COL])):

        x_chm, y_chm = to_chm_crs.transform(lon, lat)
        buffer_geom = Point(x_chm, y_chm).buffer(BUFFER_RADIUS_M)

        try:
            out_image, out_transform = rio_mask(
                chm_src, [mapping(buffer_geom)], crop=True, filled=True, nodata=chm_nodata
            )
        except ValueError:
            n_outside += 1
            continue

        chm_values = out_image[0]

        if chm_nodata is not None:
            valid_values = chm_values[chm_values != chm_nodata]
        else:
            valid_values = chm_values.flatten()

        valid_values = valid_values[~np.isnan(valid_values)]

        if valid_values.size == 0:
            n_nodata += 1
            continue

        # Compute all four percentiles from the same extracted buffer
        for p in PERCENTILES:
            uav_pct[p][i] = np.percentile(valid_values, p)

        if (i + 1) % 500 == 0:
            print(f"  Processed {i + 1:,} / {n_total:,} footprints...")

for p in PERCENTILES:
    gdf_footprints_clipped[f'p{p}_uav_chm'] = uav_pct[p]

print(f"\nUAV percentile extraction complete.")
print(f"  Total footprints              : {n_total:,}")
print(f"  Footprints outside CHM extent : {n_outside:,}")
print(f"  Footprints with no valid data : {n_nodata:,}")
print(f"  Footprints successfully done  : {n_total - n_outside - n_nodata:,}")


# ════════════════════════════════════════════════════════════════════════════
# STEP H3 — Compute GEDI vs UAV comparison stats for each RH percentile
# ════════════════════════════════════════════════════════════════════════════
comparison_rows = []

for p in PERCENTILES:
    gedi_col = GEDI_COLS[p]
    uav_col = f'p{p}_uav_chm'

    paired = gdf_footprints_clipped[[gedi_col, uav_col]].dropna()
    gedi_vals = paired[gedi_col].values
    uav_vals = paired[uav_col].values

    diff = gedi_vals - uav_vals
    bias = np.mean(diff)
    median = np.median(diff)
    rmse = np.sqrt(np.mean(diff**2))
    mae = np.mean(np.abs(diff))
    r = np.corrcoef(gedi_vals, uav_vals)[0, 1] if len(paired) > 1 else np.nan
    r2 = r**2 if not np.isnan(r) else np.nan

    comparison_rows.append({
        'rh_percentile': f'RH{p}',
        'n_pairs': len(paired),
        'bias_gedi_minus_uav': bias,
        'median': median,
        'rmse': rmse,
        'mae': mae,
        'r': r,
        'r2': r2,
    })

    # Store per-footprint difference column too
    gdf_footprints_clipped[f'diff_rh{p}_gedi_minus_uav'] = gdf_footprints_clipped[gedi_col] - gdf_footprints_clipped[uav_col]

comparison_summary = pd.DataFrame(comparison_rows)

print("\nGEDI vs UAV comparison summary:")
display(comparison_summary)


# ════════════════════════════════════════════════════════════════════════════
# STEP H4 — Save results to Excel and GeoJSON
# ════════════════════════════════════════════════════════════════════════════
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    gdf_footprints_clipped.drop(columns='geometry', errors='ignore').to_excel(
        writer, sheet_name='gedi_vs_uav_original_coords', index=False
    )
    comparison_summary.to_excel(writer, sheet_name='rh_comparison_stats', index=False)

print(f"\nExcel file updated with RH95/RH98/RH99/RH100 comparison: {excel_path}")

## Code for recreating the Shifted Iterrations DOne in the MOuntains of ERror paper


In [ ]:
# First set some controls to ensure we are in the right coordinate reference system  for our calculations
WGS84      = 'EPSG:4326'
METRIC_CRS = 'EPSG:32610'    # UTM Zone 10N — covers central California
                              # change this if your study area is in a different UTM zone

to_metric = Transformer.from_crs(WGS84, METRIC_CRS, always_xy=True)
to_wgs84  = Transformer.from_crs(METRIC_CRS, WGS84,  always_xy=True)

# For each real GEDI footprint, we simulate N_SIMULATIONS possible "true"
# positions by applying random position errors. This models the uncertainty
# in where the laser actually hit the ground vs where GEDI says it did.

bearings_degrees = np.array([0, 45, 90, 135, 180, 225, 270, 315])  # degrees
bearings_radians = np.radians(bearings_degrees)  # convert to radians for trigonometry 
offsets = np.array([5, 10, 15, 20])

records = []
for fp_idx, row in gdf_points_clipped.iterrows():

    footprint_id = row['shot_number']


    # Step 1: original unshifted position in UTM meters
    lon_orig = row.geometry.x
    lat_orig = row.geometry.y
    x_m, y_m = to_metric.transform(lon_orig, lat_orig)

    #store the original unshifted position data
    records.append({
        'footprint_id'   : footprint_id,              
        'offset'         : 0,     
        'bearing_deg'    : np.nan,
        'dx_m'           : 0.0,  
        'dy_m'           : 0.0,
        'lon_original'   : lon_orig,              
        'lat_original'   : lat_orig,        
        'lon_shifted'    : lon_orig,   
        'lat_shifted'    : lat_orig,
        'geometry'       : Point(lon_orig, lat_orig),
    })

    # loop for 32 generating all shifted positions
    for d in offsets:
        for theta_deg, theta_rad in zip(bearings_degrees, bearings_radians):

            # Step 2: compute the shifted position in UTM meters
            dx = d * np.cos(theta_rad)
            dy = d * np.sin(theta_rad)
            x_shifted = x_m + dx
            y_shifted = y_m + dy

            # Step 3: project the shifted position back to WGS84 (degrees)
            lon_shifted, lat_shifted = to_wgs84.transform(x_shifted, y_shifted)

            # Step 4: store each shifted position as a row in the results list
            records.append({
                'footprint_id'   : footprint_id,              
                'offset'         : d,     
                'bearing_deg'    : theta_deg,
                'dx_m'           : dx,  
                'dy_m'           : dy,
                'lon_original'   : lon_orig,              
                'lat_original'   : lat_orig,        
                'lon_shifted'    : lon_shifted,     
                'lat_shifted'    : lat_shifted,
                'geometry'       : Point(lon_shifted, lat_shifted),
            })

gdf_shift = gpd.GeoDataFrame(records, geometry ='geometry', crs=WGS84)

print(f"Generated {len(gdf_shift):,} rows"
      f"({gdf_shift['footprint_id'].nunique():,} footprints x "
      f"{len(offsets)*len(bearings_degrees)+1} positions for each: 1 original + 32 shifted")

# Extract UAV P95 at each of the 33 positions, then compute εd,θ

BUFFER_RADIUS_M = 12.5   # 25 m GEDI footprint diameter → 12.5 m radius buffer

with rasterio.open(chm_path) as chm_src:
    chm_nodata = chm_src.nodata
    to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)

    p95_uav = np.full(len(gdf_shift), np.nan)

    for i, (lon, lat) in enumerate(zip(gdf_shift['lon_shifted'], gdf_shift['lat_shifted'])): # giving a number to each pair of shifted coordinates
        x_chm, y_chm = to_chm_crs.transform(lon, lat) # converts each points coordinates into the CHM's CRS
        buffer_geom = Point(x_chm, y_chm).buffer(BUFFER_RADIUS_M) # draws a 12.5m buffer around each point

        try:
            out_image, _ = rio_mask( # clips the raster down to just the pixels within the circle (12.5m radius)
                chm_src, [mapping(buffer_geom)], crop=True, filled=True, nodata=chm_nodata
            )
        except ValueError:
            continue

        vals = out_image[0] # extracts all the pixel values from the clipped raster
        vals = vals[vals != chm_nodata] if chm_nodata is not None else vals.flatten() # flattens the array and removes any nodata values
        vals = vals[~np.isnan(vals)] # removes any NaN values

        if vals.size:
            p95_uav[i] = np.percentile(vals, 95) #actual P95 calculation (value below which 95% of the CHm heights in the buffer fall)
        
        if (i + 1) % 500 == 0: # every 500 iterations, print a progress updatef
            print(f"  Processed {i + 1:,} / {len(gdf_shift):,} positions...")
        
        gdf_shift['p95_uav_chm'] = p95_uav # adds the P95 values to the gdf_shift GeoDataFrame

        #attach GEDI rh95 per footprint, then compute 
        gedi_rh95_lookup = gdf_points_clipped.set_index('shot_number')['rh95'] # creates a lookup table for the GEDI RH95 values based on the shot_number
        gdf_shift['rh95_gedi'] = gdf_shift['footprint_id'].map(gedi_rh95_lookup) # maps the GEDI RH95 values to the gdf_shift GeoDataFrame based on the footprint_id

        gdf_shift['epsilon_d_theta'] = gdf_shift['p95_uav_chm'] - gdf_shift['rh95_gedi'] # computes the difference between the UAV P95 and the GEDI RH95 for each shifted position

        print(gdf_shift[['footprint_id', 'offset', 'bearing_deg', 'p95_uav_chm', 'rh95_gedi', 'epsilon_d_theta']].head(10))

with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    gdf_shift.drop(columns='geometry', errors='ignore').to_excel(
        writer, sheet_name='shift_experiment_all', index=False
    )

print(f"\nExcel file updated with shift-experiment results: {excel_path}")

# ════════════════════════════════════════════════════════════════════════════
# STEP J — Find the best-matching shifted coordinate per footprint
# ════════════════════════════════════════════════════════════════════════════
# For each footprint, identify which of the 33 positions (1 original + 32
# shifted) produced a UAV P95 value closest to the GEDI RH95 — i.e. the
# offset distance/bearing that minimizes |epsilon_d_theta|.

gdf_shift['abs_epsilon'] = gdf_shift['epsilon_d_theta'].abs()

# For each footprint_id, grab the row with the minimum absolute epsilon
best_matches = (
    gdf_shift
    .loc[gdf_shift.groupby('footprint_id')['abs_epsilon'].idxmin()]
    .reset_index(drop=True)
)

print(f"Best-matching shifted coordinate found for {len(best_matches):,} footprints.\n")

for _, fp in best_matches.iterrows():
    print(
        f"Footprint {fp['footprint_id']}: "
        f"best offset = {fp['offset']:.0f} m at bearing {fp['bearing_deg']:.0f}°  |  "
        f"shifted coord = ({fp['lon_shifted']:.6f}, {fp['lat_shifted']:.6f})  |  "
        f"GEDI RH95 = {fp['rh95_gedi']:.2f} m, UAV P95 = {fp['p95_uav_chm']:.2f} m, "
        f"error = {fp['epsilon_d_theta']:+.2f} m"
    )
# ════════════════════════════════════════════════════════════════════════════
# STEP K — Grid plot: original vs. best-matching shifted buffer, per footprint
# ════════════════════════════════════════════════════════════════════════════
from rasterio.windows import from_bounds as window_from_bounds
from matplotlib.patches import Circle
import matplotlib.pyplot as plt

n_footprints = len(best_matches)
n_cols = min(3, n_footprints)
n_rows = int(np.ceil(n_footprints / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5.5 * n_rows))
axes = np.atleast_1d(axes).flatten()

with rasterio.open(chm_path) as chm_src:
    chm_crs = chm_src.crs
    to_chm_crs = Transformer.from_crs(WGS84, chm_crs, always_xy=True)

    for ax_idx, (_, fp) in enumerate(best_matches.iterrows()):
        ax = axes[ax_idx]

        # Convert both the original and best-match points into CHM CRS
        x_orig, y_orig = to_chm_crs.transform(fp['lon_original'], fp['lat_original'])
        x_best, y_best = to_chm_crs.transform(fp['lon_shifted'], fp['lat_shifted'])

        # Build a display window that comfortably contains both buffers
        pad = BUFFER_RADIUS_M * 2.5
        min_x = min(x_orig, x_best) - pad
        max_x = max(x_orig, x_best) + pad
        min_y = min(y_orig, y_best) - pad
        max_y = max(y_orig, y_best) + pad

        window = window_from_bounds(min_x, min_y, max_x, max_y, transform=chm_src.transform)
        chm_crop = chm_src.read(1, window=window)
        crop_transform = chm_src.window_transform(window)

        # Display extent for imshow, matching the cropped raster's bounds
        extent = (
            crop_transform.c,
            crop_transform.c + chm_crop.shape[1] * crop_transform.a,
            crop_transform.f + chm_crop.shape[0] * crop_transform.e,
            crop_transform.f,
        )

        im = ax.imshow(chm_crop, extent=extent, cmap='YlGn', origin='upper')

        # Original footprint buffer (red)
        ax.add_patch(Circle((x_orig, y_orig), BUFFER_RADIUS_M,
                             fill=False, edgecolor='red', linewidth=2, label='Original'))
        ax.plot(x_orig, y_orig, 'r+', markersize=10, markeredgewidth=2)

        # Best-matching shifted buffer (blue)
        ax.add_patch(Circle((x_best, y_best), BUFFER_RADIUS_M,
                             fill=False, edgecolor='blue', linewidth=2, label='Best match'))
        ax.plot(x_best, y_best, 'b+', markersize=10, markeredgewidth=2)

        bearing_str = f"{fp['bearing_deg']:.0f}°" if pd.notna(fp['bearing_deg']) else "n/a"

        ax.set_title(
            f"Footprint {fp['footprint_id']}  |  offset={fp['offset']:.0f}m  bearing={bearing_str}\n"
            f"GEDI RH95={fp['rh95_gedi']:.2f}m  UAV P95={fp['p95_uav_chm']:.2f}m  "
            f"error={fp['epsilon_d_theta']:+.2f}m",
            fontsize=10
        )
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
        ax.legend(fontsize=7, loc='upper right')
        ax.set_aspect('equal')

# Hide any unused subplot axes if grid is not fully filled
for ax_idx in range(n_footprints, len(axes)):
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(data_folder, f'{file_header}_best_match_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Best-match grid figure saved.")